# 📊 Отчёт по пролонгациям аккаунт-менеджеров — 2023 год

**Задача:** рассчитать коэффициенты пролонгации договоров для каждого менеджера отдела сопровождения клиентов и для всего отдела в целом — за каждый месяц 2023 года и за год.

---

## 📁 Входные данные

| Файл | Описание |
|------|----------|
| `prolongations.csv` | Справочник проектов: id, месяц завершения, ФИО менеджера (AM) |
| `financial_data.csv` | Отгрузки по проектам по месяцам |

## 📤 Результат

Excel-файл `Тестовое Topface Media - Анфиса Ганнова.xlsx` с 3 листами:
1. **Весь отдел** — K1 и K2 по месяцам + итог за год
2. **Менеджеры за год** — K1 и K2 каждого менеджера за год
3. **Менеджеры по месяцам (K1+K2)** — матрица менеджер × месяц

---
## Шаг 0. Установка библиотек

In [1]:
# Все библиотеки входят в стандартный Python / Google Colab
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("✅ Библиотеки загружены")

✅ Библиотеки загружены


---
## Шаг 1. Загрузка данных

Загружаем оба файла и смотрим их структуру.

In [2]:
# ── Укажи пути к файлам ──────────────────────────────────────────────────────
PATH_PROL = "prolongations.csv"       # справочник проектов
PATH_FIN  = "financial_data.csv"      # отгрузки по месяцам
PATH_OUT  = "Тестовое Topface Media - Анфиса Ганнова.xlsx"  # куда сохранить результат

# ── Загрузка ──────────────────────────────────────────────────────────────────
prol = pd.read_csv(PATH_PROL)
fin  = pd.read_csv(PATH_FIN)

# Колонки с месяцами — всё кроме служебных полей
month_cols = [c for c in fin.columns if c not in ['id', 'Причина дубля', 'Account']]

print("=== prolongations.csv ===")
print(f"Строк: {prol.shape[0]}, уникальных проектов: {prol['id'].nunique()}")
display(prol.head(5))

print("\n=== financial_data.csv ===")
print(f"Строк: {fin.shape[0]}, уникальных проектов: {fin['id'].nunique()}")
print(f"Месяцы: {month_cols}")
display(fin.head(5))

=== prolongations.csv ===
Строк: 477, уникальных проектов: 313


,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна



=== financial_data.csv ===
Строк: 451, уникальных проектов: 314
Месяцы: ['Ноябрь 2022', 'Декабрь 2022', 'Январь 2023', 'Февраль 2023', 'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023', 'Июль 2023', 'Август 2023', 'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023', 'Январь 2024', 'Февраль 2024']


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


---
## Шаг 2. Исключение проектов со значениями СТОП / END

Если хотя бы одна ячейка проекта содержит `стоп` или `end` — проект исключается **полностью** из всех расчётов.

> Логика: `стоп`/`end` означает досрочное прекращение — это не провал менеджера, а особый случай.

In [3]:
bad_ids = set()

for col in month_cols:
    mask = fin[col].astype(str).str.strip().str.lower().isin(['стоп', 'end'])
    bad_ids.update(fin.loc[mask, 'id'].tolist())

print(f"Исключено проектов со стоп/end: {len(bad_ids)}")
print(f"Примеры исключённых id: {list(bad_ids)[:10]}")

# Убираем плохие проекты из финансовых данных
fin_clean = fin[~fin['id'].isin(bad_ids)].copy()
print(f"\nОсталось строк в financial_data: {len(fin_clean)} (было {len(fin)})")

Исключено проектов со стоп/end: 55
Примеры исключённых id: [898, 643, 900, 519, 657, 914, 788, 663, 921, 418]

Осталось строк в financial_data: 374 (было 451)


---
## Шаг 3. Запоминаем ячейки «В НОЛЬ»

`в ноль` = отгрузка формально 0, но для расчёта коэффициента нужно взять **предыдущий месяц**.

Запоминаем **до** преобразования в числа — иначе потеряем эту информацию.

In [4]:
v_nol = set()  # множество пар (id проекта, колонка-месяц)

for col in month_cols:
    mask = fin_clean[col].astype(str).str.strip().str.lower() == 'в ноль'
    for pid in fin_clean.loc[mask, 'id']:
        v_nol.add((pid, col))

print(f"Ячеек «в ноль» найдено: {len(v_nol)}")
print("Примеры:", list(v_nol)[:5])

Ячеек «в ноль» найдено: 105
Примеры: [(887, 'Август 2023'), (579, 'Ноябрь 2022'), (789, 'Май 2023'), (346, 'Июль 2023'), (818, 'Январь 2024')]


---
## Шаг 4. Парсинг чисел

Числа в CSV хранятся как строки в русском формате: `"1 234,56"` (пробел = тысячи, запятая = дробь).

Преобразуем в `float`. Всё нечисловое → `NaN`.

In [5]:
def parse_val(v):
    """Преобразует строку '1 234,56' в float. Нечисловые значения → NaN."""
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

for col in month_cols:
    fin_clean[col] = fin_clean[col].apply(parse_val)

print("✅ Числа распарсены")
display(fin_clean[month_cols].head(3))

✅ Числа распарсены


,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024
0,36220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,38045.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## Шаг 5. Суммирование дублей

Один проект может иметь несколько строк в `financial_data` (первая/вторая часть оплаты и т.д.).

Группируем по `id` и **суммируем** отгрузки за каждый месяц.

> `min_count=1` — если все значения NaN, результат тоже NaN (а не 0)

In [6]:
fin_sum = fin_clean.groupby('id')[month_cols].sum(min_count=1).reset_index()

print(f"До суммирования: {len(fin_clean)} строк")
print(f"После суммирования: {len(fin_sum)} строк (уникальные проекты)")
display(fin_sum.head(3))

До суммирования: 374 строк
После суммирования: 259 строк (уникальные проекты)


,id,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024
0,15,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,31,55100.0,55100.0,NaN,44775.00,44775.00,44775.0,44775.0,44775.00,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,46200.0


---
## Шаг 6. Применяем логику «В НОЛЬ» → предыдущий месяц

Для проектов с `в ноль`: если итоговая сумма за месяц = 0 или NaN, заменяем на значение **предыдущего месяца**.

In [7]:
replaced_count = 0

for i, col in enumerate(month_cols):
    if i == 0:
        continue  # у первого месяца нет предыдущего
    prev_col = month_cols[i - 1]
    ids_to_fix = [pid for (pid, c) in v_nol if c == col]
    
    for pid in ids_to_fix:
        rows = fin_sum[fin_sum['id'] == pid]
        if rows.empty:
            continue
        idx = rows.index[0]
        cur = fin_sum.at[idx, col]
        if pd.isna(cur) or cur == 0:
            fin_sum.at[idx, col] = fin_sum.at[idx, prev_col]
            replaced_count += 1

print(f"✅ Заменено ячеек «в ноль» → предыдущий месяц: {replaced_count}")

✅ Заменено ячеек «в ноль» → предыдущий месяц: 22


---
## Шаг 7. Вспомогательные функции

Создаём удобные инструменты для расчёта.

In [8]:
# Словарь для быстрого доступа: id → {месяц: сумма}
fin_dict = fin_sum.set_index('id')[month_cols].to_dict(orient='index')

def get_ship(project_id, col_name):
    """Возвращает отгрузку проекта за месяц. None если данных нет."""
    if project_id not in fin_dict:
        return None
    v = fin_dict[project_id].get(col_name)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return v

def prev_month(m):
    """Возвращает предыдущий месяц (M-1)."""
    i = month_cols.index(m)
    return month_cols[i - 1] if i > 0 else None

def prev2_month(m):
    """Возвращает позапрошлый месяц (M-2)."""
    i = month_cols.index(m)
    return month_cols[i - 2] if i > 1 else None

print("✅ Вспомогательные функции готовы")

✅ Вспомогательные функции готовы


---
## Шаг 8. Основной расчёт — K1 и K2 за каждый месяц 2023

Для каждого месяца 2023 года считаем оба коэффициента по каждому проекту.

**Важные нюансы:**
- Месяцы в `prolongations` — строчными буквами (`январь 2023`), в `financial_data` — с заглавной (`Январь 2023`) → приводим к одному регистру
- Менеджер берётся из колонки `AM` в `prolongations` по **конкретной строке цикла** (у одного проекта в разных циклах может быть разный менеджер)
- Проекты из `bad_ids` пропускаем

In [9]:
months_2023 = [
    'Январь 2023', 'Февраль 2023', 'Март 2023',    'Апрель 2023',
    'Май 2023',    'Июнь 2023',    'Июль 2023',    'Август 2023',
    'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023'
]

records = []  # каждая запись = один проект в одном цикле пролонгации

for M in months_2023:
    M1 = prev_month(M)    # M-1 (заглавный, формат financial_data)
    M2 = prev2_month(M)   # M-2

    M1_low = M1.lower() if M1 else None  # строчный (формат prolongations)
    M2_low = M2.lower() if M2 else None

    # ── K1: проекты завершившиеся в M-1 ──────────────────────────────────────
    if M1_low and M1_low in prol['month'].values:
        for _, row in prol[prol['month'] == M1_low].iterrows():
            pid = int(row['id'])
            am  = row['AM']  # менеджер из prolongations (приоритетный источник)

            if pid in bad_ids:
                continue

            ship_m1 = get_ship(pid, M1)  # знаменатель: отгрузка за M-1
            ship_m  = get_ship(pid, M)   # числитель:   отгрузка за M

            if ship_m1 is None or ship_m1 == 0:
                continue  # нет базы для расчёта

            prolonged = 1 if (ship_m is not None and ship_m > 0) else 0

            records.append({
                'month': M, 'AM': am, 'coef': 'K1',
                'denom': ship_m1,
                'numer': ship_m if prolonged else 0,
                'prolonged': prolonged
            })

    # ── K2: проекты завершившиеся в M-2, НЕ продлившиеся в M-1 ──────────────
    if M2_low and M1_low:
        # Проекты, которые УЖЕ продлились в M-1 — исключаем из K2
        if M1_low in prol['month'].values:
            ids_prol_m1 = set(prol[prol['month'] == M1_low]['id'].astype(int))
        else:
            ids_prol_m1 = set()

        for _, row in prol[prol['month'] == M2_low].iterrows():
            pid = int(row['id'])
            am  = row['AM']

            if pid in bad_ids:
                continue
            if pid in ids_prol_m1:
                continue  # уже пролонгирован в первый месяц — не берём

            ship_m2 = get_ship(pid, M2)  # знаменатель: отгрузка за M-2
            ship_m  = get_ship(pid, M)   # числитель:   отгрузка за M

            if ship_m2 is None or ship_m2 == 0:
                continue

            prolonged = 1 if (ship_m is not None and ship_m > 0) else 0

            records.append({
                'month': M, 'AM': am, 'coef': 'K2',
                'denom': ship_m2,
                'numer': ship_m if prolonged else 0,
                'prolonged': prolonged
            })

df = pd.DataFrame(records)
managers = sorted(df['AM'].unique())

print(f"✅ Расчёт завершён")
print(f"Всего записей: {len(df)}")
print(f"Менеджеры ({len(managers)}): {managers}")
display(df.head(10))

✅ Расчёт завершён
Всего записей: 549
Менеджеры (8): ['Васильев Артем Александрович', 'Иванова Мария Сергеевна', 'Кузнецов Михаил Иванович', 'Михайлов Андрей Сергеевич', 'Петрова Анна Дмитриевна', 'Попова Екатерина Николаевна', 'Смирнова Ольга Владимировна', 'Соколова Анастасия Викторовна']


,month,AM,coef,denom,numer,prolonged
0,Январь 2023,Иванова Мария Сергеевна,K1,439280.00,102433.75,1
1,Январь 2023,Иванова Мария Сергеевна,K1,55195.00,0.00,0
2,Январь 2023,Васильев Артем Александрович,K1,55100.00,0.00,0
3,Январь 2023,Соколова Анастасия Викторовна,K1,137562.50,83901.00,1
4,Январь 2023,Смирнова Ольга Владимировна,K1,57615.00,61925.00,1
5,Январь 2023,Михайлов Андрей Сергеевич,K1,210015.21,201944.09,1
6,Январь 2023,Васильев Артем Александрович,K1,501895.00,660475.00,1
7,Январь 2023,Васильев Артем Александрович,K1,602370.00,0.00,0
8,Январь 2023,Иванова Мария Сергеевна,K1,137700.00,149206.50,1
9,Январь 2023,Иванова Мария Сергеевна,K1,110700.00,0.00,0


---
## Шаг 9. Сборка итоговых таблиц

In [10]:
def calc_coef(data):
    """
    Коэффициент = сумма числителей / сумма знаменателей.
    НЕ среднее коэффициентов — это важно для корректного итога за год.
    """
    d = data['denom'].sum()
    n = data['numer'].sum()
    return round(n / d, 4) if d > 0 else None

# ── Таблица 1: Весь отдел по месяцам ─────────────────────────────────────────
t1_rows = []
for M in months_2023 + ['Итого за год']:
    row = {'Месяц': M}
    for k in ['K1', 'K2']:
        sub = df[df['coef'] == k] if M == 'Итого за год' else df[(df['month'] == M) & (df['coef'] == k)]
        d = sub['denom'].sum(); n = sub['numer'].sum()
        suffix = '(K1)' if k == 'K1' else '(K2)'
        num    = '1' if k == 'K1' else '2'
        row[f'К пролонгации {suffix}']  = round(d, 2)
        row[f'Пролонгировано {suffix}'] = round(n, 2)
        row[f'Коэффициент {num}']       = calc_coef(sub)
    t1_rows.append(row)

table1 = pd.DataFrame(t1_rows, columns=['Месяц',
    'К пролонгации (K1)', 'Пролонгировано (K1)', 'Коэффициент 1',
    'К пролонгации (K2)', 'Пролонгировано (K2)', 'Коэффициент 2'])

print("=== Таблица 1: Весь отдел ===")
display(table1)

# ── Таблица 2: Менеджеры за год ───────────────────────────────────────────────
t2_rows = []
for am in managers + ['Весь отдел']:
    row = {'Менеджер': am}
    for k in ['K1', 'K2']:
        sub = df[df['coef'] == k] if am == 'Весь отдел' else df[(df['AM'] == am) & (df['coef'] == k)]
        d = sub['denom'].sum(); n = sub['numer'].sum()
        suffix = '(K1)' if k == 'K1' else '(K2)'
        num    = '1' if k == 'K1' else '2'
        row[f'К пролонгации {suffix}']  = round(d, 2)
        row[f'Пролонгировано {suffix}'] = round(n, 2)
        row[f'Коэффициент {num}']       = calc_coef(sub)
    t2_rows.append(row)

table2 = pd.DataFrame(t2_rows, columns=['Менеджер',
    'К пролонгации (K1)', 'Пролонгировано (K1)', 'Коэффициент 1',
    'К пролонгации (K2)', 'Пролонгировано (K2)', 'Коэффициент 2'])

print("\n=== Таблица 2: Менеджеры за год ===")
display(table2)

# ── Таблицы 3 и 4: Менеджеры по месяцам ──────────────────────────────────────
def build_monthly_table(coef_key):
    rows = []
    for am in managers + ['Весь отдел']:
        row = {'Менеджер': am}
        for M in months_2023:
            sub = df[(df['month'] == M) & (df['coef'] == coef_key)] if am == 'Весь отдел' \
                  else df[(df['AM'] == am) & (df['month'] == M) & (df['coef'] == coef_key)]
            row[M] = calc_coef(sub)
        sub_y = df[df['coef'] == coef_key] if am == 'Весь отдел' \
                else df[(df['AM'] == am) & (df['coef'] == coef_key)]
        row['Итого за год'] = calc_coef(sub_y)
        rows.append(row)
    return pd.DataFrame(rows)

table3_k1 = build_monthly_table('K1')
table3_k2 = build_monthly_table('K2')

print("\n=== Таблица 3: K1 по месяцам ===")
display(table3_k1)
print("\n=== Таблица 4: K2 по месяцам ===")
display(table3_k2)

=== Таблица 1: Весь отдел ===


,Месяц,К пролонгации (K1),Пролонгировано (K1),Коэффициент 1,К пролонгации (K2),Пролонгировано (K2),Коэффициент 2
0,Январь 2023,5918646.86,2696551.61,0.4556,1856379.00,1466110.00,0.7898
1,Февраль 2023,2531100.50,1880945.00,0.7431,5220516.86,1942100.03,0.3720
2,Март 2023,1636367.77,1029859.40,0.6294,2230565.50,1297074.08,0.5815
3,Апрель 2023,2858072.20,1026683.35,0.3592,1393554.60,866511.00,0.6218
4,Май 2023,3052776.50,1497140.75,0.4904,2638533.45,829448.44,0.3144
5,Июнь 2023,1127180.53,281673.28,0.2499,3052776.50,1711009.00,0.5605
6,Июль 2023,2579457.85,1348911.00,0.5229,1127180.53,363425.00,0.3224
7,Август 2023,1936915.83,943678.88,0.4872,2211122.85,933925.32,0.4224
8,Сентябрь 2023,2981878.26,954859.42,0.3202,1443195.83,620866.25,0.4302
9,Октябрь 2023,3356820.81,2670570.19,0.7956,2791110.62,833185.96,0.2985



=== Таблица 2: Менеджеры за год ===


,Менеджер,К пролонгации (K1),Пролонгировано (K1),Коэффициент 1,К пролонгации (K2),Пролонгировано (K2),Коэффициент 2
0,Васильев Артем Александрович,11012707.79,5261484.43,0.4778,10195717.37,4949259.85,0.4854
1,Иванова Мария Сергеевна,4448267.16,1528450.55,0.3436,4295639.66,1100125.25,0.2561
2,Кузнецов Михаил Иванович,816474.44,470182.98,0.5759,130902.53,165407.39,1.2636
3,Михайлов Андрей Сергеевич,3361833.59,2235422.29,0.6649,4170398.59,2307729.98,0.5534
4,Петрова Анна Дмитриевна,98492.00,109442.52,1.1112,0.00,0.00,NaN
5,Попова Екатерина Николаевна,2817918.08,1223950.80,0.4343,2910733.08,1449293.77,0.4979
6,Смирнова Ольга Владимировна,2817219.59,1925523.50,0.6835,2078410.05,1750145.25,0.8421
7,Соколова Анастасия Викторовна,6242489.34,3646137.97,0.5841,4521725.24,2546856.00,0.5632
8,Весь отдел,31615401.99,16400595.04,0.5188,28303526.52,14268817.49,0.5041



=== Таблица 3: K1 по месяцам ===


,Менеджер,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Итого за год
0,Васильев Артем Александрович,0.5982,1.0566,0.6272,0.1281,0.3277,0.3893,0.5850,0.4775,0.1724,0.8868,0.6210,0.3241,0.4778
1,Иванова Мария Сергеевна,0.2701,NaN,0.4461,0.2777,1.0000,0.0000,0.4726,0.5407,1.0000,NaN,NaN,NaN,0.3436
2,Кузнецов Михаил Иванович,1.2660,0.8470,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,1.3085,0.9534,0.2740,0.5759
3,Михайлов Андрей Сергеевич,0.6879,0.9036,2.4306,1.0988,1.1896,0.0000,1.1975,NaN,0.0000,0.4491,NaN,0.0000,0.6649
4,Петрова Анна Дмитриевна,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.1112,1.1112
5,Попова Екатерина Николаевна,0.4407,0.0000,0.4963,0.1663,0.6051,0.0000,0.0891,0.4799,0.7470,0.7002,NaN,0.0000,0.4343
6,Смирнова Ольга Владимировна,0.7434,0.0000,NaN,0.9661,NaN,0.0000,0.0000,0.3725,0.7655,1.0384,0.8362,0.3611,0.6835
7,Соколова Анастасия Викторовна,0.4008,0.2769,0.9210,1.0658,0.1953,0.7664,0.6225,0.5413,0.2602,0.8152,0.2706,0.7911,0.5841
8,Весь отдел,0.4556,0.7431,0.6294,0.3592,0.4904,0.2499,0.5229,0.4872,0.3202,0.7956,0.6394,0.5315,0.5188



=== Таблица 4: K2 по месяцам ===


,Менеджер,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Итого за год
0,Васильев Артем Александрович,0.5278,0.2586,1.1872,0.6058,0.0995,0.4736,0.4517,0.4868,0.5256,0.2300,0.8878,0.8175,0.4854
1,Иванова Мария Сергеевна,0.0000,0.2062,NaN,0.0000,0.1627,0.7414,0.0000,0.5053,0.3881,1.0000,NaN,NaN,0.2561
2,Кузнецов Михаил Иванович,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,1.4467,1.1905,1.2636
3,Михайлов Андрей Сергеевич,1.0480,0.7379,0.1838,0.9963,1.1920,1.0253,0.0000,0.0000,NaN,0.0000,0.3749,NaN,0.5534
4,Петрова Анна Дмитриевна,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Попова Екатерина Николаевна,0.2958,0.5041,0.0000,0.7881,0.3395,0.6906,0.0000,0.1648,0.4842,0.7470,0.7141,NaN,0.4979
6,Смирнова Ольга Владимировна,NaN,0.7310,1.1505,NaN,1.0363,NaN,0.5163,1.5531,0.4803,0.6761,1.0413,0.7580,0.8421
7,Соколова Анастасия Викторовна,0.9688,0.5844,0.3608,0.8956,0.9560,0.2488,0.6955,0.4264,0.0000,0.0000,0.9153,0.2747,0.5632
8,Весь отдел,0.7898,0.3720,0.5815,0.6218,0.3144,0.5605,0.3224,0.4224,0.4302,0.2985,0.8093,0.7011,0.5041


---
## Шаг 10. Оформление и сохранение Excel-файла

Создаём итоговый Excel с форматированием по шаблону:
- 🟡 Жёлтые заголовки
- Объединённые ячейки для групп K1 и K2  
- Коэффициенты в формате процентов
- Суммы с разделителем тысяч
- Прочерк «—» вместо пустых значений

In [11]:
# ── Стили ────────────────────────────────────────────────────────────────────
YELLOW       = "FFD966"
YELLOW_LIGHT = "FFF2CC"
GRAY_ROW     = "F9F9F9"
FONT_NAME    = "Arial"

fill_yellow       = PatternFill("solid", fgColor=YELLOW)
fill_yellow_light = PatternFill("solid", fgColor=YELLOW_LIGHT)
fill_white        = PatternFill("solid", fgColor="FFFFFF")
fill_gray         = PatternFill("solid", fgColor=GRAY_ROW)
fill_total        = PatternFill("solid", fgColor="FFE699")

thin = Side(style="thin", color="BFBFBF")

def font_bold(size=10): return Font(name=FONT_NAME, bold=True, size=size)
def font_reg(size=10, color="000000"): return Font(name=FONT_NAME, size=size, color=color)
def align_c(): return Alignment(horizontal="center", vertical="center", wrap_text=True)
def align_l(): return Alignment(horizontal="left",   vertical="center", wrap_text=True)
def brd(): return Border(left=thin, right=thin, top=thin, bottom=thin)

def hdr(cell, value, fill):
    cell.value = value; cell.font = font_bold()
    cell.fill = fill; cell.alignment = align_c(); cell.border = brd()

def dat(cell, value, fill, fmt=None, bold=False):
    is_nan = value is None or (isinstance(value, float) and np.isnan(value))
    if is_nan:
        cell.value = "—"; cell.font = font_reg(color="999999")
    elif fmt == "pct":
        cell.value = value; cell.number_format = "0.0%"
        cell.font = font_bold() if bold else font_reg()
    elif fmt == "num":
        cell.value = value; cell.number_format = "#,##0"
        cell.font = font_bold() if bold else font_reg()
    else:
        cell.value = value; cell.font = font_bold() if bold else font_reg()
    cell.fill = fill; cell.alignment = align_c(); cell.border = brd()

print("✅ Стили определены")

✅ Стили определены


In [12]:
wb = Workbook()
wb.remove(wb.active)

months_short = ['Янв','Фев','Мар','Апр','Май','Июн',
                'Июл','Авг','Сен','Окт','Ноя','Дек']

# ═══════════════════════════════════════════════════════════════════════
# ЛИСТ 1: Весь отдел
# ═══════════════════════════════════════════════════════════════════════
ws1 = wb.create_sheet("Весь отдел")
ws1.freeze_panes = "B4"

ws1.merge_cells("A1:G1")
hdr(ws1["A1"], "Отчёт по пролонгациям — Весь отдел — 2023 год", fill_yellow)
ws1["A1"].font = font_bold(12)
ws1.row_dimensions[1].height = 22

ws1.merge_cells("A2:A3"); hdr(ws1["A2"], "Месяц", fill_yellow_light)
ws1.merge_cells("B2:D2"); hdr(ws1["B2"], "Пролонгации в первый месяц (K1)", fill_yellow)
ws1.merge_cells("E2:G2"); hdr(ws1["E2"], "Пролонгации через месяц (K2)", fill_yellow)
ws1.row_dimensions[2].height = 20

for i, h in enumerate(["К пролонгации","Пролонгировано","Коэффициент"]*2):
    hdr(ws1.cell(row=3, column=i+2), h, fill_yellow_light)
ws1.row_dimensions[3].height = 30

for ri, row in table1.iterrows():
    er = ri + 4
    is_tot = row['Месяц'] == 'Итого за год'
    rf = fill_total if is_tot else (fill_gray if ri % 2 == 0 else fill_white)
    c = ws1.cell(row=er, column=1)
    c.value = row['Месяц']; c.font = font_bold() if is_tot else font_reg()
    c.fill = fill_yellow_light if is_tot else rf
    c.alignment = align_l(); c.border = brd()
    for ci, (col, fmt) in enumerate([
        ('К пролонгации (K1)','num'),('Пролонгировано (K1)','num'),('Коэффициент 1','pct'),
        ('К пролонгации (K2)','num'),('Пролонгировано (K2)','num'),('Коэффициент 2','pct')]):
        dat(ws1.cell(row=er, column=ci+2), row[col],
            fill_yellow_light if is_tot else rf, fmt=fmt, bold=is_tot)

ws1.column_dimensions['A'].width = 18
for col in ['B','C','D','E','F','G']: ws1.column_dimensions[col].width = 16

# ═══════════════════════════════════════════════════════════════════════
# ЛИСТ 2: Менеджеры за год
# ═══════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("Менеджеры за год")
ws2.freeze_panes = "B4"

ws2.merge_cells("A1:G1")
hdr(ws2["A1"], "Отчёт по пролонгациям — Менеджеры — Итоги 2023 года", fill_yellow)
ws2["A1"].font = font_bold(12)
ws2.row_dimensions[1].height = 22

ws2.merge_cells("A2:A3"); hdr(ws2["A2"], "Менеджер", fill_yellow_light)
ws2.merge_cells("B2:D2"); hdr(ws2["B2"], "Пролонгации в первый месяц (K1)", fill_yellow)
ws2.merge_cells("E2:G2"); hdr(ws2["E2"], "Пролонгации через месяц (K2)", fill_yellow)
ws2.row_dimensions[2].height = 20

for i, h in enumerate(["К пролонгации","Пролонгировано","Коэффициент"]*2):
    hdr(ws2.cell(row=3, column=i+2), h, fill_yellow_light)
ws2.row_dimensions[3].height = 30

for ri, row in table2.iterrows():
    er = ri + 4
    is_tot = row['Менеджер'] == 'Весь отдел'
    rf = fill_total if is_tot else (fill_gray if ri % 2 == 0 else fill_white)
    c = ws2.cell(row=er, column=1)
    c.value = row['Менеджер']; c.font = font_bold() if is_tot else font_reg()
    c.fill = fill_yellow_light if is_tot else rf
    c.alignment = align_l(); c.border = brd()
    for ci, (col, fmt) in enumerate([
        ('К пролонгации (K1)','num'),('Пролонгировано (K1)','num'),('Коэффициент 1','pct'),
        ('К пролонгации (K2)','num'),('Пролонгировано (K2)','num'),('Коэффициент 2','pct')]):
        dat(ws2.cell(row=er, column=ci+2), row[col],
            fill_yellow_light if is_tot else rf, fmt=fmt, bold=is_tot)

ws2.column_dimensions['A'].width = 32
for col in ['B','C','D','E','F','G']: ws2.column_dimensions[col].width = 16

# ═══════════════════════════════════════════════════════════════════════
# ЛИСТЫ 3 и 4: Менеджеры по месяцам
# ═══════════════════════════════════════════════════════════════════════
def build_monthly_sheet(wb, sheet_name, df_m, coef_label):
    ws = wb.create_sheet(sheet_name)
    ws.freeze_panes = "B4"
    last = get_column_letter(14)

    ws.merge_cells(f"A1:{last}1")
    hdr(ws["A1"], f"Отчёт по пролонгациям — {coef_label} — По месяцам — 2023 год", fill_yellow)
    ws["A1"].font = font_bold(12); ws.row_dimensions[1].height = 22

    ws.merge_cells("A2:A3"); hdr(ws["A2"], "Менеджер", fill_yellow_light)
    ws.merge_cells("B2:M2"); hdr(ws["B2"], f"{coef_label} по месяцам 2023", fill_yellow)
    ws.merge_cells("N2:N3"); hdr(ws["N2"], "Итого за год", fill_yellow)
    ws.row_dimensions[2].height = 20

    for i, m in enumerate(months_short):
        hdr(ws.cell(row=3, column=i+2), m, fill_yellow_light)
    ws.row_dimensions[3].height = 20

    for ri, row in df_m.iterrows():
        er = ri + 4
        is_tot = row['Менеджер'] == 'Весь отдел'
        rf = fill_total if is_tot else (fill_gray if ri % 2 == 0 else fill_white)
        c = ws.cell(row=er, column=1)
        c.value = row['Менеджер']; c.font = font_bold() if is_tot else font_reg()
        c.fill = fill_yellow_light if is_tot else rf
        c.alignment = align_l(); c.border = brd()
        for mi, month in enumerate(months_2023):
            dat(ws.cell(row=er, column=mi+2), row.get(month),
                fill_yellow_light if is_tot else rf, fmt="pct", bold=is_tot)
        dat(ws.cell(row=er, column=14), row.get('Итого за год'),
            fill_yellow_light if is_tot else rf, fmt="pct", bold=is_tot)

    ws.column_dimensions['A'].width = 32
    for i in range(2, 15): ws.column_dimensions[get_column_letter(i)].width = 9

build_monthly_sheet(wb, "Менеджеры К1 по месяцам", table3_k1, "Коэффициент 1 (K1)")
build_monthly_sheet(wb, "Менеджеры К2 по месяцам", table3_k2, "Коэффициент 2 (K2)")

wb.save(PATH_OUT)
print(f"✅ Файл сохранён: {PATH_OUT}")
print(f"\nСводка:")
print(f"  Проектов исключено (стоп/end): {len(bad_ids)}")
print(f"  Ячеек «в ноль» обработано:    {len(v_nol)}")
print(f"  Менеджеров в отчёте:          {len(managers)}")
print(f"  Листов в Excel:               4")

✅ Файл сохранён: Тестовое Topface Media - Анфиса Ганнова.xlsx

Сводка:
  Проектов исключено (стоп/end): 55
  Ячеек «в ноль» обработано:    105
  Менеджеров в отчёте:          8
  Листов в Excel:               4


---
## ✅ Готово!

Файл `Тестовое Topface Media - Анфиса Ганнова.xlsx` содержит 3 листа:

| Лист | Содержимое |
|------|-----------|
| **Весь отдел** | K1 и K2 по месяцам + итог за год |
| **Менеджеры за год** | K1 и K2 каждого менеджера + итог отдела |
| **Менеджеры (К1&К2) по месяцам** | Матрица: менеджер × месяц, значения = K1, Матрица: менеджер × месяц, значения = K2

### 💡 Примечание по пустым значениям (—)
Прочерк в ячейке означает, что у данного менеджера **не было проектов** для расчёта в этом месяце — это корректно и не является ошибкой.